## Практическая работа 3 — Векторные представления (C#)

- Единое предобработанное ядро текстов (нормализация, токенизация, стоп-слова)
- Три способа представления документов: LSA, Word2Vec, Doc2Vec
- Логистическая регрессия и метрики качества
- Сравнение, поиск похожих документов, визуализации

In [ ]:
#r "nuget: Microsoft.Data.Analysis, 0.22.2"
#r "nuget: Microsoft.ML, 3.0.1"
#r "nuget: Microsoft.ML.Mkl.Components, 3.0.1"
#r "nuget: ScottPlot, 5.0.56"
#r "nuget: System.Text.Encoding.CodePages, 8.0.0"

In [ ]:
using System;
using System.IO;
using System.Text;
using System.Text.RegularExpressions;
using System.Collections.Generic;
using System.Linq;
using System.Diagnostics;

using Microsoft.Data.Analysis;
using Microsoft.ML;
using Microsoft.ML.Data;
using Microsoft.ML.Transforms.Text;

using ScottPlot;
using Microsoft.DotNet.Interactive.Formatting;

Encoding.RegisterProvider(CodePagesEncodingProvider.Instance);

Formatter.Register<DataFrame>((df, writer) => writer.Write(df.ToString()), "text/plain");
Formatter.Register(typeof(ScottPlot.Plot),
    (obj, writer) => writer.Write(((ScottPlot.Plot)obj).GetPngHtml(900, 520)),
    HtmlFormatter.MimeType);

var mlContext = new MLContext(seed: 42);

double Sigmoid(double x) => 1.0 / (1.0 + Math.Exp(-x));

double Cosine(float[] a, float[] b)
{
    double dot = 0, na = 0, nb = 0;
    int len = Math.Min(a.Length, b.Length);
    for (int i = 0; i < len; i++)
    {
        double aa = a[i];
        double bb = b[i];
        dot += aa * bb;
        na += aa * aa;
        nb += bb * bb;
    }
    return na > 0 && nb > 0 ? dot / Math.Sqrt(na * nb) : 0.0;
}

void NormalizeInPlace(float[] vector)
{
    double sumSq = 0;
    for (int i = 0; i < vector.Length; i++)
        sumSq += vector[i] * vector[i];
    if (sumSq <= 0) return;
    float scale = (float)(1.0 / Math.Sqrt(sumSq));
    for (int i = 0; i < vector.Length; i++)
        vector[i] *= scale;
}

float[] Clone(float[] source)
{
    var copy = new float[source.Length];
    Array.Copy(source, copy, source.Length);
    return copy;
}

float[] ToDense(VBuffer<float> buffer)
{
    var dense = new float[buffer.Length];
    buffer.CopyTo(dense);
    return dense;
}

int SampleNegative(Random random, double[] cumulative)
{
    var value = random.NextDouble();
    int idx = Array.BinarySearch(cumulative, value);
    if (idx < 0) idx = ~idx;
    if (idx >= cumulative.Length) idx = cumulative.Length - 1;
    return idx;
}

void Shuffle<T>(IList<T> list, Random random)
{
    for (int i = list.Count - 1; i > 0; i--)
    {
        int j = random.Next(i + 1);
        (list[i], list[j]) = (list[j], list[i]);
    }
}

string Truncate(string text, int maxLength = 160)
{
    if (string.IsNullOrWhiteSpace(text)) return string.Empty;
    return text.Length <= maxLength ? text : text.Substring(0, maxLength) + "…";
}

List<(int index, double score)> GetTopSimilar(float[] query, IReadOnlyList<float[]> candidates, int top = 5)
{
    var list = new List<(int index, double score)>(candidates.Count);
    for (int i = 0; i < candidates.Count; i++)
    {
        var score = Cosine(query, candidates[i]);
        if (double.IsNaN(score))
            continue;
        list.Add((index: i, score: score));
    }
    return list
        .OrderByDescending(item => item.score)
        .Take(top)
        .ToList();
}

int[] SampleIndices(int length, int maxSample, Random random)
{
    if (length <= maxSample)
        return Enumerable.Range(0, length).ToArray();

    var reservoir = new int[maxSample];
    for (int i = 0; i < maxSample; i++)
        reservoir[i] = i;

    for (int i = maxSample; i < length; i++)
    {
        int j = random.Next(i + 1);
        if (j < maxSample)
            reservoir[j] = i;
    }

    Array.Sort(reservoir);
    return reservoir;
}


### 1. Загрузка и первичная очистка

In [ ]:
DataFrame LoadDf(string path, string encoding = "windows-1251")
{
    using var fs = File.OpenRead(path);
    return DataFrame.LoadCsv(fs, separator: ',', header: true, guessRows: 50_000, addIndexColumn: false, encoding: Encoding.GetEncoding(encoding));
}

var dfTrainRaw = LoadDf("../data/train.csv");
var dfTestRaw  = LoadDf("../data/test.csv");

DataFrame Clean(DataFrame df)
{
    var sentimentCol = df.Columns["sentiment"] as StringDataFrameColumn;
    var notNeutral = sentimentCol.ElementwiseNotEquals("neutral");
    return df.Filter(notNeutral);
}

var dfTrain = Clean(dfTrainRaw);
var dfTest  = Clean(dfTestRaw);

Console.WriteLine($"Train raw: {dfTrainRaw.Rows.Count}, after filter: {dfTrain.Rows.Count}");
Console.WriteLine($"Test  raw: {dfTestRaw.Rows.Count}, after filter: {dfTest.Rows.Count}");

dfTrain.Head(3)


### 2. Нормализация, токенизация и корпус

In [ ]:
public class SentimentRecord
{
    public int DocId { get; set; }
    public string Text { get; set; }
    public string Sentiment { get; set; }
    public bool Label { get; set; }
    public string Split { get; set; }
}

public class TokenizedRecord : SentimentRecord
{
    public string NormalizedText { get; set; }
    public ReadOnlyMemory<char>[] TokensRaw { get; set; }
    public ReadOnlyMemory<char>[] Tokens { get; set; }
}

public class ProcessedRecord
{
    public int GlobalId { get; set; }
    public bool IsTrain { get; set; }
    public int SplitIndex { get; set; }
    public string Text { get; set; }
    public string CleanText { get; set; }
    public string Sentiment { get; set; }
    public bool Label { get; set; }
    public string[] Tokens { get; set; }
}

var allRecords = new List<SentimentRecord>();
int docCounter = 0;

void AddRecords(DataFrame df, string split)
{
    var textCol = df.Columns["text"];
    var sentimentCol = df.Columns["sentiment"];
    for (int i = 0; i < df.Rows.Count; i++)
    {
        var text = textCol[i]?.ToString();
        var sentiment = sentimentCol[i]?.ToString();
        if (string.IsNullOrWhiteSpace(text) || string.IsNullOrWhiteSpace(sentiment))
            continue;
        allRecords.Add(new SentimentRecord
        {
            DocId = docCounter++,
            Text = text,
            Sentiment = sentiment,
            Label = sentiment.Equals("positive", StringComparison.OrdinalIgnoreCase),
            Split = split
        });
    }
}

AddRecords(dfTrain, "train");
AddRecords(dfTest,  "test");

Console.WriteLine($"Records after cleanup: {allRecords.Count} (train: {allRecords.Count(r => r.Split == "train")}, test: {allRecords.Count(r => r.Split == "test")})");

var allDataView = mlContext.Data.LoadFromEnumerable(allRecords);
var textPipeline = mlContext.Transforms.Text.NormalizeText(
        outputColumnName: "NormalizedText",
        inputColumnName: nameof(SentimentRecord.Text),
        caseMode: TextNormalizingEstimator.CaseMode.Lower,
        keepDiacritics: false,
        keepNumbers: false,
        keepPunctuations: false)
    .Append(mlContext.Transforms.Text.TokenizeIntoWords(
        outputColumnName: "TokensRaw",
        inputColumnName: "NormalizedText"))
    .Append(mlContext.Transforms.Text.RemoveDefaultStopWords(
        outputColumnName: "Tokens",
        inputColumnName: "TokensRaw",
        language: StopWordsRemovingEstimator.Language.English));

var textTransformer = textPipeline.Fit(allDataView);
var tokenizedView = textTransformer.Transform(allDataView);
var tokenized = mlContext.Data.CreateEnumerable<TokenizedRecord>(tokenizedView, reuseRowObject: false).ToList();

List<ProcessedRecord> processedRecords = new();
List<ProcessedRecord> trainRecords = new();
List<ProcessedRecord> testRecords = new();

foreach (var rec in tokenized)
{
    var tokens = (rec.Tokens ?? Array.Empty<ReadOnlyMemory<char>>())
        .Select(t => t.ToString())
        .Where(t => !string.IsNullOrWhiteSpace(t) && t.Length > 1)
        .ToArray();

    if (tokens.Length < 3)
        continue;

    bool isTrain = rec.Split == "train";
    var processed = new ProcessedRecord
    {
        IsTrain = isTrain,
        Text = rec.Text,
        CleanText = string.Join(" ", tokens),
        Sentiment = rec.Sentiment,
        Label = rec.Label,
        Tokens = tokens
    };
    processedRecords.Add(processed);
    if (isTrain)
        trainRecords.Add(processed);
    else
        testRecords.Add(processed);
}

int maxTrainDocs = 4000;
if (trainRecords.Count > maxTrainDocs)
{
    var rng = new Random(42);
    Shuffle(trainRecords, rng);
    trainRecords = trainRecords.Take(maxTrainDocs).ToList();
}

var reordered = new List<ProcessedRecord>();
int globalIdx = 0;
for (int i = 0; i < trainRecords.Count; i++)
{
    var rec = trainRecords[i];
    rec.GlobalId = globalIdx++;
    rec.SplitIndex = i;
    rec.IsTrain = true;
    reordered.Add(rec);
}
for (int i = 0; i < testRecords.Count; i++)
{
    var rec = testRecords[i];
    rec.GlobalId = globalIdx++;
    rec.SplitIndex = i;
    rec.IsTrain = false;
    reordered.Add(rec);
}
processedRecords = reordered;

Console.WriteLine($"Train docs after preprocessing: {trainRecords.Count}");
Console.WriteLine($"Test  docs after preprocessing: {testRecords.Count}");

foreach (var example in trainRecords.Take(3))
{
    Console.WriteLine("---");
    Console.WriteLine(example.Text);
    Console.WriteLine($"Clean → {example.CleanText}");
}

public class ModelInput
{
    public string Text { get; set; }
    public bool Label { get; set; }
}

public class VectorExample
{
    public bool Label { get; set; }
    [VectorType(128)]
    public float[] Features { get; set; }
}

var trainInputs = trainRecords.Select(r => new ModelInput { Text = r.CleanText, Label = r.Label }).ToList();
var testInputs  = testRecords.Select(r => new ModelInput { Text = r.CleanText, Label = r.Label }).ToList();

var embeddingTrainer = mlContext.BinaryClassification.Trainers.LbfgsLogisticRegression(
    labelColumnName: nameof(VectorExample.Label),
    featureColumnName: nameof(VectorExample.Features));

BinaryClassificationMetrics? metricsTrainWord2Vec = null;
BinaryClassificationMetrics? metricsTestWord2Vec = null;
BinaryClassificationMetrics? metricsTrainDoc2Vec = null;
BinaryClassificationMetrics? metricsTestDoc2Vec = null;
float[][] word2VecTrainMatrix = Array.Empty<float[]>();
float[][] word2VecTestMatrix = Array.Empty<float[]>();
float[][] doc2VecTrainMatrix = Array.Empty<float[]>();
float[][] doc2VecTestMatrix = Array.Empty<float[]>();



### 3. LSA: TF-IDF + PCA + нормализация

In [ ]:
int lsaRank = 300;

var allInputs = trainInputs.Concat(testInputs).ToList();
var allInputsView = mlContext.Data.LoadFromEnumerable(allInputs);

var lsaPipeline = mlContext.Transforms.Text.FeaturizeText(
        outputColumnName: "FeaturesRaw",
        inputColumnName: nameof(ModelInput.Text))
    .Append(mlContext.Transforms.ProjectToPrincipalComponents(
        outputColumnName: "FeaturesPca",
        inputColumnName: "FeaturesRaw",
        rank: lsaRank,
        ensureZeroMean: false))
    .Append(mlContext.Transforms.NormalizeLpNorm(
        outputColumnName: "Features",
        inputColumnName: "FeaturesPca"));

var lsaTransformer = lsaPipeline.Fit(allInputsView);

var trainLsaView = lsaTransformer.Transform(mlContext.Data.LoadFromEnumerable(trainInputs));
var testLsaView  = lsaTransformer.Transform(mlContext.Data.LoadFromEnumerable(testInputs));

var lsaTrainer = mlContext.BinaryClassification.Trainers.LbfgsLogisticRegression(
    labelColumnName: nameof(ModelInput.Label),
    featureColumnName: "Features");

var lsaModel = lsaTrainer.Fit(trainLsaView);
var predTrainLsa = lsaModel.Transform(trainLsaView);
var predTestLsa  = lsaModel.Transform(testLsaView);

var metricsTrainLsa = mlContext.BinaryClassification.Evaluate(predTrainLsa, labelColumnName: nameof(ModelInput.Label));
var metricsTestLsa  = mlContext.BinaryClassification.Evaluate(predTestLsa, labelColumnName: nameof(ModelInput.Label));

Console.WriteLine("LSA — train metrics:");
Console.WriteLine($"  Accuracy: {metricsTrainLsa.Accuracy:F4}, F1: {metricsTrainLsa.F1Score:F4}, AUC: {metricsTrainLsa.AreaUnderRocCurve:F4}");
Console.WriteLine("LSA — test metrics:");
Console.WriteLine($"  Accuracy: {metricsTestLsa.Accuracy:F4}, F1: {metricsTestLsa.F1Score:F4}, AUC: {metricsTestLsa.AreaUnderRocCurve:F4}");

public class FeatureRow
{
    public bool Label { get; set; }
    public VBuffer<float> Features { get; set; }
}

public class FeatureWithPca : FeatureRow
{
    public VBuffer<float> FeaturesPca { get; set; }
}

var lsaTrainVectors = mlContext.Data.CreateEnumerable<FeatureRow>(trainLsaView, reuseRowObject: false)
    .Select(r => ToDense(r.Features))
    .ToArray();
var lsaTestVectors = mlContext.Data.CreateEnumerable<FeatureRow>(testLsaView, reuseRowObject: false)
    .Select(r => ToDense(r.Features))
    .ToArray();

var lsaTrainPca = mlContext.Data.CreateEnumerable<FeatureWithPca>(trainLsaView, reuseRowObject: false)
    .ToArray();


### 4. Word2Vec: Skip-gram + Negative Sampling

In [ ]:
Dictionary<string, int> wordToId = default!;
double[] negativeSampler = Array.Empty<double>();
int[][] docTokenIds = Array.Empty<int[]>();
int[][] docTokenIdsSampled = Array.Empty<int[]>();

try
{
    var tokenCounts = new Dictionary<string, int>(StringComparer.Ordinal);
    foreach (var doc in processedRecords)
    {
        foreach (var token in doc.Tokens)
        {
            if (!tokenCounts.TryGetValue(token, out var cnt)) cnt = 0;
            tokenCounts[token] = cnt + 1;
        }
    }

    int minCount = 5;
    var vocab = tokenCounts
        .Where(kv => kv.Value >= minCount)
        .OrderByDescending(kv => kv.Value)
        .Select((kv, idx) => new { Token = kv.Key, Count = kv.Value, Index = idx })
        .ToArray();

    wordToId = vocab.ToDictionary(v => v.Token, v => v.Index, StringComparer.Ordinal);

    Console.WriteLine($"Word2Vec vocab size: {wordToId.Count}");

    var freqPow = vocab.Select(v => Math.Pow(v.Count, 0.75)).ToArray();
    double totalPow = freqPow.Sum();
    negativeSampler = new double[freqPow.Length];
    double accum = 0;
    for (int i = 0; i < negativeSampler.Length; i++)
    {
        accum += freqPow[i] / totalPow;
        negativeSampler[i] = accum;
    }
    if (negativeSampler.Length > 0)
        negativeSampler[^1] = 1.0;

    docTokenIds = processedRecords
        .Select(doc => doc.Tokens
            .Select(tok => wordToId.TryGetValue(tok, out var id) ? id : -1)
            .Where(id => id >= 0)
            .ToArray())
        .ToArray();

    int maxTokensPerDoc = 80;
    var rngSubsample = new Random(2024);
    docTokenIdsSampled = docTokenIds
        .Select(tokens =>
        {
            var indices = SampleIndices(tokens.Length, maxTokensPerDoc, rngSubsample);
            return indices.Select(idx => tokens[idx]).ToArray();
        })
        .ToArray();

    int embeddingSize = 128;
    int window = 4;
    int negativeSamples = 4;
    int word2VecEpochs = 2;
    double initialLr = 0.025;

    var rngWord2Vec = new Random(42);

    var inputEmbeddings = new float[wordToId.Count][];
    var outputEmbeddings = new float[wordToId.Count][];
    for (int i = 0; i < wordToId.Count; i++)
    {
        inputEmbeddings[i] = new float[embeddingSize];
        outputEmbeddings[i] = new float[embeddingSize];
        for (int d = 0; d < embeddingSize; d++)
        {
            inputEmbeddings[i][d] = (float)((rngWord2Vec.NextDouble() - 0.5) / embeddingSize);
            outputEmbeddings[i][d] = 0f;
        }
    }

    var trainingDocIndices = docTokenIdsSampled
        .Select((tokens, idx) => (tokens, idx))
        .Where(item => item.tokens.Length >= 2)
        .Select(item => item.idx)
        .ToList();

    if (trainingDocIndices.Count > 8000)
    {
        Shuffle(trainingDocIndices, rngWord2Vec);
        trainingDocIndices = trainingDocIndices.Take(8000).ToList();
    }

    for (int epoch = 0; epoch < word2VecEpochs; epoch++)
    {
        var order = trainingDocIndices.ToList();
        Shuffle(order, rngWord2Vec);

        double loss = 0;
        double lr = initialLr * (1.0 - (double)epoch / Math.Max(1, word2VecEpochs));
        lr = Math.Max(lr, initialLr * 0.25);

        foreach (var docIdx in order)
        {
            var tokens = docTokenIdsSampled[docIdx];
            if (tokens.Length < 2)
                continue;

            for (int pos = 0; pos < tokens.Length; pos++)
            {
                int centerId = tokens[pos];
                if (centerId < 0) continue;

                int currentWindow = rngWord2Vec.Next(1, window + 1);
                int start = Math.Max(0, pos - currentWindow);
                int end = Math.Min(tokens.Length - 1, pos + currentWindow);

                for (int ctx = start; ctx <= end; ctx++)
                {
                    if (ctx == pos) continue;
                    int contextId = tokens[ctx];
                    if (contextId < 0) continue;

                    var centerVec = inputEmbeddings[centerId];
                    var contextVec = outputEmbeddings[contextId];

                    double score = 0;
                    for (int d = 0; d < embeddingSize; d++)
                        score += centerVec[d] * contextVec[d];

                    double sig = Sigmoid(score);
                    double grad = (1.0 - sig) * lr;

                    for (int d = 0; d < embeddingSize; d++)
                    {
                        float cVal = centerVec[d];
                        float oVal = contextVec[d];
                        float deltaCenter = (float)(grad * oVal);
                        float deltaContext = (float)(grad * cVal);
                        centerVec[d] += deltaCenter;
                        contextVec[d] += deltaContext;
                    }
                    loss -= Math.Log(sig + 1e-8);

                    for (int n = 0; n < negativeSamples; n++)
                    {
                        int negativeId = SampleNegative(rngWord2Vec, negativeSampler);
                        if (negativeId == centerId || negativeId == contextId)
                            continue;

                        var negativeVec = outputEmbeddings[negativeId];
                        double negScore = 0;
                        for (int d = 0; d < embeddingSize; d++)
                            negScore += centerVec[d] * negativeVec[d];

                        double negSig = Sigmoid(negScore);
                        double negGrad = (-negSig) * lr;

                        for (int d = 0; d < embeddingSize; d++)
                        {
                            float cVal = centerVec[d];
                            float nVal = negativeVec[d];
                            float deltaCenter = (float)(negGrad * nVal);
                            float deltaNeg = (float)(negGrad * cVal);
                            centerVec[d] += deltaCenter;
                            negativeVec[d] += deltaNeg;
                        }

                        loss -= Math.Log(1.0 - negSig + 1e-8);
                    }
                }
            }
        }

        Console.WriteLine($"Word2Vec epoch {epoch + 1}/{word2VecEpochs} — loss ≈ {loss:F2}, lr={lr:F4}");
    }

    float[] BuildWord2VecDoc(int[] tokenIds)
    {
        var vector = new float[embeddingSize];
        int count = 0;
        foreach (var id in tokenIds)
        {
            if (id < 0 || id >= inputEmbeddings.Length) continue;
            var embed = inputEmbeddings[id];
            for (int d = 0; d < embeddingSize; d++)
                vector[d] += embed[d];
            count++;
        }
        if (count == 0)
            return vector;
        float inv = 1f / count;
        for (int d = 0; d < embeddingSize; d++)
            vector[d] *= inv;
        NormalizeInPlace(vector);
        return vector;
    }

    var word2VecDocVectors = docTokenIds.Select(BuildWord2VecDoc).ToArray();
    word2VecTrainMatrix = trainRecords.Select(r => word2VecDocVectors[r.GlobalId]).ToArray();
    word2VecTestMatrix  = testRecords.Select(r => word2VecDocVectors[r.GlobalId]).ToArray();

    var word2VecTrainExamples = trainRecords
        .Select(r => new VectorExample { Label = r.Label, Features = Clone(word2VecDocVectors[r.GlobalId]) })
        .ToList();
    var word2VecTestExamples = testRecords
        .Select(r => new VectorExample { Label = r.Label, Features = Clone(word2VecDocVectors[r.GlobalId]) })
        .ToList();

    var word2VecTrainView = mlContext.Data.LoadFromEnumerable(word2VecTrainExamples);
    var word2VecTestView  = mlContext.Data.LoadFromEnumerable(word2VecTestExamples);

    var word2VecModel = embeddingTrainer.Fit(word2VecTrainView);
    var predTrainWord2Vec = word2VecModel.Transform(word2VecTrainView);
    var predTestWord2Vec  = word2VecModel.Transform(word2VecTestView);

    metricsTrainWord2Vec = mlContext.BinaryClassification.Evaluate(predTrainWord2Vec, labelColumnName: nameof(VectorExample.Label));
    var metricsTestWord2Vec  = mlContext.BinaryClassification.Evaluate(predTestWord2Vec, labelColumnName: nameof(VectorExample.Label));

    Console.WriteLine("Word2Vec — train metrics:");
    Console.WriteLine($"  Accuracy: {metricsTrainWord2Vec.Accuracy:F4}, F1: {metricsTrainWord2Vec.F1Score:F4}, AUC: {metricsTrainWord2Vec.AreaUnderRocCurve:F4}");
    Console.WriteLine("Word2Vec — test metrics:");
    Console.WriteLine($"  Accuracy: {metricsTestWord2Vec.Accuracy:F4}, F1: {metricsTestWord2Vec.F1Score:F4}, AUC: {metricsTestWord2Vec.AreaUnderRocCurve:F4}");
}
catch (Exception ex)
{
    Console.WriteLine($"Word2Vec error: {ex}");
    throw;
}


### 5. Doc2Vec (Distributed Bag of Words)

In [ ]:
try
{
    int doc2VecDim = 128;
    int doc2VecEpochs = 3;
    int doc2VecNegative = 4;
    double doc2VecLr = 0.05;

    var docEmbeddings = new float[processedRecords.Count][];
    var docWordOutput = new float[wordToId.Count][];
    var rngDoc2Vec = new Random(99);

    for (int i = 0; i < docEmbeddings.Length; i++)
    {
        docEmbeddings[i] = new float[doc2VecDim];
        for (int d = 0; d < doc2VecDim; d++)
            docEmbeddings[i][d] = (float)((rngDoc2Vec.NextDouble() - 0.5) / doc2VecDim);
    }
    for (int i = 0; i < docWordOutput.Length; i++)
    {
        docWordOutput[i] = new float[doc2VecDim];
        for (int d = 0; d < doc2VecDim; d++)
            docWordOutput[i][d] = (float)((rngDoc2Vec.NextDouble() - 0.5) / doc2VecDim);
    }

    for (int epoch = 0; epoch < doc2VecEpochs; epoch++)
    {
        var order = Enumerable.Range(0, processedRecords.Count).ToList();
        Shuffle(order, rngDoc2Vec);

        double loss = 0;
        double lr = doc2VecLr * (1.0 - (double)epoch / Math.Max(1, doc2VecEpochs));
        lr = Math.Max(lr, doc2VecLr * 0.3);

        foreach (var docIdx in order)
        {
            var tokens = docTokenIdsSampled[docIdx];
            if (tokens.Length == 0)
                continue;

            var docVec = docEmbeddings[docIdx];

            foreach (var wordId in tokens)
            {
                var wordVec = docWordOutput[wordId];
                double score = 0;
                for (int d = 0; d < doc2VecDim; d++)
                    score += docVec[d] * wordVec[d];

                double sig = Sigmoid(score);
                double grad = (1.0 - sig) * lr;

                for (int d = 0; d < doc2VecDim; d++)
                {
                    float dv = docVec[d];
                    float wv = wordVec[d];
                    float deltaDoc = (float)(grad * wv);
                    float deltaWord = (float)(grad * dv);
                    docVec[d] += deltaDoc;
                    wordVec[d] += deltaWord;
                }
                loss -= Math.Log(sig + 1e-8);

                for (int n = 0; n < doc2VecNegative; n++)
                {
                    int negativeId = SampleNegative(rngDoc2Vec, negativeSampler);
                    if (negativeId == wordId)
                        continue;

                    var negativeVec = docWordOutput[negativeId];
                    double negScore = 0;
                    for (int d = 0; d < doc2VecDim; d++)
                        negScore += docVec[d] * negativeVec[d];

                    double negSig = Sigmoid(negScore);
                    double negGrad = (-negSig) * lr;

                    for (int d = 0; d < doc2VecDim; d++)
                    {
                        float dv = docVec[d];
                        float nv = negativeVec[d];
                        float deltaDoc = (float)(negGrad * nv);
                        float deltaNeg = (float)(negGrad * dv);
                        docVec[d] += deltaDoc;
                        negativeVec[d] += deltaNeg;
                    }

                    loss -= Math.Log(1.0 - negSig + 1e-8);
                }
            }
        }

        Console.WriteLine($"Doc2Vec epoch {epoch + 1}/{doc2VecEpochs} — loss ≈ {loss:F2}, lr={lr:F4}");
    }

    foreach (var docVec in docEmbeddings)
        NormalizeInPlace(docVec);

    doc2VecTrainMatrix = trainRecords.Select(r => docEmbeddings[r.GlobalId]).ToArray();
    doc2VecTestMatrix  = testRecords.Select(r => docEmbeddings[r.GlobalId]).ToArray();
var doc2VecTrainExamples = trainRecords
        .Select(r => new VectorExample { Label = r.Label, Features = Clone(docEmbeddings[r.GlobalId]) })
        .ToList();
    var doc2VecTestExamples = testRecords
        .Select(r => new VectorExample { Label = r.Label, Features = Clone(docEmbeddings[r.GlobalId]) })
        .ToList();

    var doc2VecTrainView = mlContext.Data.LoadFromEnumerable(doc2VecTrainExamples);
    var doc2VecTestView  = mlContext.Data.LoadFromEnumerable(doc2VecTestExamples);

    var doc2VecModel = embeddingTrainer.Fit(doc2VecTrainView);
    var predTrainDoc2Vec = doc2VecModel.Transform(doc2VecTrainView);
    var predTestDoc2Vec  = doc2VecModel.Transform(doc2VecTestView);

    metricsTrainDoc2Vec = mlContext.BinaryClassification.Evaluate(predTrainDoc2Vec, labelColumnName: nameof(VectorExample.Label));
    metricsTestDoc2Vec  = mlContext.BinaryClassification.Evaluate(predTestDoc2Vec, labelColumnName: nameof(VectorExample.Label));

    Console.WriteLine("Doc2Vec — train metrics:");
    Console.WriteLine($"  Accuracy: {metricsTrainDoc2Vec.Accuracy:F4}, F1: {metricsTrainDoc2Vec.F1Score:F4}, AUC: {metricsTrainDoc2Vec.AreaUnderRocCurve:F4}");
    Console.WriteLine("Doc2Vec — test metrics:");
    Console.WriteLine($"  Accuracy: {metricsTestDoc2Vec.Accuracy:F4}, F1: {metricsTestDoc2Vec.F1Score:F4}, AUC: {metricsTestDoc2Vec.AreaUnderRocCurve:F4}");
}
catch (Exception ex)
{
    Console.WriteLine($"Doc2Vec error: {ex}");
    throw;
}


### 6. Сравнение метрик

In [ ]:
Console.WriteLine("\nМетрики (test):");
Console.WriteLine("Method      Acc     F1     AUC");
Console.WriteLine($"LSA        {metricsTestLsa.Accuracy,6:F3}  {metricsTestLsa.F1Score,6:F3}  {metricsTestLsa.AreaUnderRocCurve,6:F3}");
if (metricsTestWord2Vec is not null)
    Console.WriteLine($"Word2Vec    {metricsTestWord2Vec.Accuracy,6:F3}  {metricsTestWord2Vec.F1Score,6:F3}  {metricsTestWord2Vec.AreaUnderRocCurve,6:F3}");
else
    Console.WriteLine("Word2Vec    n/a");
if (metricsTestDoc2Vec is not null)
    Console.WriteLine($"Doc2Vec     {metricsTestDoc2Vec.Accuracy,6:F3}  {metricsTestDoc2Vec.F1Score,6:F3}  {metricsTestDoc2Vec.AreaUnderRocCurve,6:F3}");
else
    Console.WriteLine("Doc2Vec     n/a");


### 7. Поиск похожих документов

In [ ]:
int exampleIndex = Math.Min(2, testRecords.Count - 1);
var queryDoc = testRecords[exampleIndex];

Console.WriteLine("Запрос (test):");
Console.WriteLine(queryDoc.Text);
Console.WriteLine($"Clean → {queryDoc.CleanText}");
Console.WriteLine($"Sentiment: {queryDoc.Sentiment}");

var lsaQuery = lsaTestVectors[exampleIndex];
var word2VecQuery = word2VecTestMatrix[exampleIndex];
var doc2VecQuery = doc2VecTestMatrix[exampleIndex];

void PrintNeighbours(string title, float[] query, IReadOnlyList<float[]> matrix)
{
    Console.WriteLine($"\n{title}");
    var neighbours = GetTopSimilar(query, matrix, 5);
    foreach (var (idx, score) in neighbours)
    {
        var doc = trainRecords[idx];
        Console.WriteLine($"Score {score:F3} | {doc.Sentiment}");
        Console.WriteLine(Truncate(doc.Text));
        Console.WriteLine("---");
    }
}

PrintNeighbours("LSA", lsaQuery, lsaTrainVectors);
PrintNeighbours("Word2Vec", word2VecQuery, word2VecTrainMatrix);
PrintNeighbours("Doc2Vec", doc2VecQuery, doc2VecTrainMatrix);


### 8. Визуализация

In [ ]:
// LSA: cumulative explained variance
int components = Math.Min(lsaRank, lsaTrainPca.Length > 0 ? lsaTrainPca[0].FeaturesPca.Length : 0);
var variance = new double[components];
var mean = new double[components];
long count = 0;
foreach (var row in lsaTrainPca.Take(5000))
{
    var dense = ToDense(row.FeaturesPca);
    count++;
    for (int i = 0; i < components; i++)
    {
        double val = dense[i];
        mean[i] += val;
        variance[i] += val * val;
    }
}
if (count > 0)
{
    for (int i = 0; i < components; i++)
    {
        mean[i] /= count;
        variance[i] = variance[i] / count - mean[i] * mean[i];
        if (variance[i] < 0) variance[i] = 0;
    }
}
var totalVar = variance.Sum();
var cumulative = new double[components];
double accVar = 0;
for (int i = 0; i < components; i++)
{
    double share = totalVar > 0 ? variance[i] / totalVar : 0;
    accVar += share;
    cumulative[i] = accVar * 100.0;
}

var pltVariance = new ScottPlot.Plot();
var xsVar = Enumerable.Range(1, components).Select(i => (double)i).ToArray();
pltVariance.Add.Scatter(xsVar, cumulative);
Console.WriteLine($"Variance curve prepared (components={components})");

public class TwoDimRow
{
    public bool Label { get; set; }
    public VBuffer<float> PC { get; set; }
}

TwoDimRow[] ProjectTo2D(float[][] matrix, IList<ProcessedRecord> records)
{
    try
    {
        if (matrix.Length == 0 || records.Count == 0)
            return Array.Empty<TwoDimRow>();

        int count = Math.Min(matrix.Length, records.Count);
        var data = records.Take(count).Select((rec, idx) => new VectorExample
        {
            Label = rec.Label,
            Features = Clone(matrix[idx])
        });
        var view = mlContext.Data.LoadFromEnumerable(data);
        var pca = mlContext.Transforms.ProjectToPrincipalComponents("PC", nameof(VectorExample.Features), rank: 2).Fit(view);
        return mlContext.Data.CreateEnumerable<TwoDimRow>(pca.Transform(view), reuseRowObject: false).ToArray();
    }
    catch (Exception ex)
    {
        Console.WriteLine($"ProjectTo2D error (records={records.Count}, matrix={matrix.Length}): {ex.Message}");
        return Array.Empty<TwoDimRow>();
    }
}

var lsa2D = ProjectTo2D(lsaTrainVectors, trainRecords);
var word2Vec2D = ProjectTo2D(word2VecTrainMatrix, trainRecords);
var doc2Vec2D = ProjectTo2D(doc2VecTrainMatrix, trainRecords);

void DescribeScatter(TwoDimRow[] rows, string title)
{
    var samplePos = rows.Where(r => r.Label).Take(5).Select(r => ToDense(r.PC)).ToArray();
    var sampleNeg = rows.Where(r => !r.Label).Take(5).Select(r => ToDense(r.PC)).ToArray();
    double avgPosX = samplePos.Length > 0 ? samplePos.Average(v => v[0]) : 0;
    double avgNegX = sampleNeg.Length > 0 ? sampleNeg.Average(v => v[0]) : 0;
    Console.WriteLine($"{title}: points={rows.Length}, avg PC1 pos={avgPosX:F3}, avg PC1 neg={avgNegX:F3}");
}

DescribeScatter(lsa2D, "LSA 2D");
DescribeScatter(word2Vec2D, "Word2Vec 2D");
DescribeScatter(doc2Vec2D, "Doc2Vec 2D");
